# Выбор признаков: проверяем руками

Ноутбук к теме [`lesson.md`](lesson.md). Здесь считается всё, что в лекции сказано словами:
как отбор до валидации выдаёт 90% точности на чистом шуме и как `Pipeline` это чинит;
чем взаимная информация отличается от корреляции; почему на XOR ломается не только
одномерный фильтр, но и жадная обёртка; насколько врут важности из коробки; и
насколько случаен выбор представителя из группы одинаковых признаков.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif, mutual_info_regression,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

C_BG, C_INK, C_GRAY = "#faf9f5", "#141413", "#b0aea5"
C_PANEL, C_ORANGE, C_BLUE, C_GREEN = "#e8e6dc", "#d97757", "#6a9bcc", "#788c5d"
plt.rcParams.update({
    "figure.facecolor": C_BG, "axes.facecolor": C_BG, "axes.edgecolor": C_GRAY,
    "text.color": C_INK, "xtick.color": C_INK, "ytick.color": C_INK,
    "axes.labelcolor": C_INK, "font.size": 11, "legend.frameon": False,
})

## 1. Отбор до валидации: 90% точности из ничего

Данные, в которых нет никакого сигнала: признаки независимы от меток, метки — монетка.
Правильный ответ — 0.5. Смотрим, что скажут две процедуры.

In [2]:
rng = np.random.default_rng(0)
n, p, k = 100, 5000, 100
X = rng.normal(size=(n, p))
y = rng.integers(0, 2, size=n)      # метки не связаны с X вообще

model = KNeighborsClassifier(1)

# НЕПРАВИЛЬНО: отбор один раз по всем данным, потом кросс-валидация
X_sel = SelectKBest(f_classif, k=k).fit_transform(X, y)
naive = cross_val_score(model, X_sel, y, cv=5).mean()

# ПРАВИЛЬНО: отбор — шаг пайплайна, переобучается на каждом фолде
pipe = Pipeline([("select", SelectKBest(f_classif, k=k)), ("model", model)])
honest = cross_val_score(pipe, X, y, cv=5).mean()

print(f"n = {n}, p = {p}, сигнала нет; правильный ответ — 0.5")
print(f"  отбор по всем данным, затем CV : {naive:.3f}")
print(f"  отбор внутри Pipeline          : {honest:.3f}")
print(f"\nразница {naive - honest:+.3f} — это ровно та информация о метках,")
print("которую наивная схема протащила в отбор")

n = 100, p = 5000, сигнала нет; правильный ответ — 0.5
  отбор по всем данным, затем CV : 0.880
  отбор внутри Pipeline          : 0.490

разница +0.390 — это ровно та информация о метках,
которую наивная схема протащила в отбор


Оба варианта написаны в три строки, и внешне они почти одинаковы. Разница только в том,
что во втором `SelectKBest` живёт **внутри** `Pipeline` — и переобучается на каждом фолде
заново. Это и есть весь рецепт: всё, что смотрит на `y`, обязано быть шагом пайплайна.

Отдельно стоит убедиться, что при $p \gg n$ идеально коррелирующие с шумом признаки
действительно находятся:

In [3]:
Xc = X - X.mean(axis=0)
yc = y - y.mean()
corr = (Xc.T @ yc) / np.sqrt((Xc**2).sum(axis=0) * (yc**2).sum())
print(f"максимальная |корреляция| среди {p} чистых шумов: {np.abs(corr).max():.3f}")
print(f"признаков с |corr| > 0.3: {(np.abs(corr) > 0.3).sum()}")
print(f"ожидаемый разброс корреляции при n = {n}: ±{1 / np.sqrt(n - 3):.3f}")

максимальная |корреляция| среди 5000 чистых шумов: 0.399
признаков с |corr| > 0.3: 8
ожидаемый разброс корреляции при n = 100: ±0.102


## 2. Корреляция против взаимной информации

Корреляция меряет линейную связь, взаимная информация — любую. Проверим на четырёх
зависимостях, где связь то линейная, то симметричная, то её нет вовсе.

In [4]:
rng = np.random.default_rng(1)
n = 2000
x = rng.uniform(-1, 1, size=n)

cases = {
    "линейная  y = x": x,
    "квадрат   y = x²": x**2,
    "синус     y = sin(6x)": np.sin(6 * x),
    "шум       y ⟂ x": rng.uniform(-1, 1, size=n),
}

print(f"{'зависимость':<24}{'|корреляция|':>14}{'взаимная информация':>22}")
for name, target in cases.items():
    yv = target + rng.normal(0, 0.1, size=n)
    c = abs(np.corrcoef(x, yv)[0, 1])
    mi = mutual_info_regression(x.reshape(-1, 1), yv, random_state=0)[0]
    print(f"{name:<24}{c:>14.3f}{mi:>22.3f}")

зависимость               |корреляция|   взаимная информация
линейная  y = x                  0.985                 1.674
квадрат   y = x²                 0.025                 1.024
синус     y = sin(6x)            0.388                 1.721
шум       y ⟂ x                  0.017                 0.000


На квадрате корреляция почти ноль при полной детерминированности связи — классический
контрпример, который стоит уметь привести. На синусе она тоже сильно занижена (0.39 при
почти функциональной зависимости), а MI видит эту связь не хуже линейной.

Строка с шумом дала ровно 0.000 — но не обольщайтесь: оценка MI в sklearn неотрицательна
по построению (отрицательные значения обрезаются в ноль), и на 2000 объектах у чистого
шума она обнулилась. На выборке поменьше шум получает заметно ненулевые оценки:

In [5]:
rng = np.random.default_rng(2)
print(f"{'n':>7}{'MI у шума: медиана':>22}{'максимум из 300':>18}")
for n_small in (200, 600, 2000, 8000):
    mis = np.concatenate([
        mutual_info_classif(rng.normal(size=(n_small, 50)),
                            rng.integers(0, 2, size=n_small), random_state=0)
        for _ in range(6)
    ])
    print(f"{n_small:>7}{np.median(mis):>22.4f}{mis.max():>18.4f}")
print("\nчем меньше выборка, тем шумнее оценка — и тем ненадёжнее ранжирование по MI")

      n    MI у шума: медиана   максимум из 300


    200                0.0000            0.1144


    600                0.0001            0.0499


   2000                0.0006            0.0340


   8000                0.0001            0.0148

чем меньше выборка, тем шумнее оценка — и тем ненадёжнее ранжирование по MI


Отсюда практическое правило: MI хорош как **грубое сито** на больших выборках, но
ранжировать им сотни слабых признаков при маленьком n не стоит.

## 3. XOR: ломается не только фильтр

Пара признаков, каждый из которых по отдельности независим от цели. Сравним три
процедуры отбора: одномерный фильтр, жадную обёртку (forward selection) и полный
перебор пар.

In [6]:
rng = np.random.default_rng(0)
n = 600
x1, x2 = rng.normal(size=n), rng.normal(size=n)
y = ((x1 > 0) ^ (x2 > 0)).astype(int)
decoy = np.where(y == 1, 0.45, -0.45) + rng.normal(0, 1.0, size=n)
X = np.column_stack([x1, x2, decoy, rng.normal(size=(n, 6))])
names = ["x1 (XOR)", "x2 (XOR)", "приманка"] + [f"шум {i+1}" for i in range(6)]

def score(cols, seed=0):
    '''Честная оценка набора признаков: 5-фолдовая CV случайного леса.'''
    if not len(cols):
        return 0.5
    return cross_val_score(RandomForestClassifier(n_estimators=200, random_state=seed),
                           X[:, list(cols)], y, cv=5).mean()

# --- одномерный фильтр
F = f_classif(X, y)[0]
filt = list(np.argsort(F)[::-1][:2])

# --- жадная обёртка: добавляем признак с лучшим приростом, пока прирост есть
chosen, best = [], 0.5
history = []
for _ in range(3):
    cand = [(score(chosen + [j]), j) for j in range(X.shape[1]) if j not in chosen]
    s, j = max(cand)
    history.append((names[j], s))
    if s <= best + 1e-3:
        break
    chosen, best = chosen + [j], s

# --- полный перебор пар
pairs = [((i, j), score([i, j])) for i in range(X.shape[1]) for j in range(i + 1, X.shape[1])]
best_pair, best_pair_score = max(pairs, key=lambda t: t[1])

print("одномерный фильтр, топ-2 :", [names[i] for i in filt],
      f"→ CV {score(filt):.3f}")
print("жадная обёртка, шаги     :", " → ".join(f"{nm} ({s:.3f})" for nm, s in history))
print("жадная обёртка, итог     :", [names[i] for i in chosen] or "пусто",
      f"→ CV {best:.3f}")
print("полный перебор пар       :", [names[i] for i in best_pair],
      f"→ CV {best_pair_score:.3f}")
print("все девять признаков     :", f"CV {score(range(X.shape[1])):.3f}")

одномерный фильтр, топ-2 : ['приманка', 'шум 6'] → CV 0.595
жадная обёртка, шаги     : шум 6 (0.540) → приманка (0.615) → x2 (XOR) (0.643)
жадная обёртка, итог     : ['шум 6', 'приманка', 'x2 (XOR)'] → CV 0.643
полный перебор пар       : ['x1 (XOR)', 'x2 (XOR)'] → CV 0.992


все девять признаков     : CV 0.868


## 4. Важности признаков: MDI против permutation importance

Три информативных признака, огрублённых до трёх категорий, плюс непрерывный шум
(все значения разные) и бинарный шум. Смотрим, кому достанется важность.

In [7]:
n = 800
X_inf, y = make_classification(n_samples=n, n_features=3, n_informative=3,
                               n_redundant=0, class_sep=1.4, random_state=5)
rng = np.random.default_rng(5)
X_cat = np.stack([np.digitize(X_inf[:, j], np.quantile(X_inf[:, j], [0.33, 0.66]))
                  for j in range(3)], axis=1).astype(float)
X = np.column_stack([X_cat, rng.normal(size=n), rng.integers(0, 2, size=n)])
names = ["инф. 1 (3 кат.)", "инф. 2 (3 кат.)", "инф. 3 (3 кат.)",
         "шум непрерывный", "шум бинарный"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr)

mdi = rf.feature_importances_
p_tr = permutation_importance(rf, Xtr, ytr, n_repeats=20, random_state=0).importances_mean
p_te = permutation_importance(rf, Xte, yte, n_repeats=20, random_state=0).importances_mean

print(f"качество леса: обучение {rf.score(Xtr, ytr):.3f}, тест {rf.score(Xte, yte):.3f}\n")
print(f"{'признак':<20}{'MDI':>9}{'perm (обуч.)':>15}{'perm (тест)':>14}")
for nm, a, b, c in zip(names, mdi, p_tr, p_te):
    print(f"{nm:<20}{a:>9.3f}{b:>15.3f}{c:>14.3f}")
print(f"\nдоля MDI у чистого непрерывного шума: {100 * mdi[3] / mdi.sum():.0f}%")
print(f"он же по permutation на тесте:        {p_te[3]:+.3f}")

качество леса: обучение 1.000, тест 0.944

признак                   MDI   perm (обуч.)   perm (тест)
инф. 1 (3 кат.)         0.138          0.185         0.157
инф. 2 (3 кат.)         0.118          0.178         0.159
инф. 3 (3 кат.)         0.575          0.360         0.327
шум непрерывный         0.158          0.058         0.000
шум бинарный            0.011          0.028        -0.008

доля MDI у чистого непрерывного шума: 16%
он же по permutation на тесте:        +0.000


Непрерывный шум обгоняет по MDI два настоящих признака: у него в каждом узле сотни
возможных порогов, и среди них найдётся случайно удачный. На отложенной части
permutation importance ставит его на честный ноль. Обратите внимание и на средний
столбец: permutation, посчитанная на **обучающих** данных, шуму тоже кое-что даёт —
переобученный лес действительно использовал его для запоминания.

## 5. Устойчивость: кого выберет Lasso из группы близнецов

Пять признаков с корреляцией 0.99 и одинаковым вкладом. Прогоняем отбор на бутстрэп-
повторах и считаем, как часто каждый попадает в модель.

In [8]:
rng = np.random.default_rng(4)
n, alpha, l1_ratio, R = 120, 1.0, 0.5, 200

base = rng.normal(size=n)
group = base[:, None] + rng.normal(0, 0.10, size=(n, 5))
X = np.column_stack([group, rng.normal(size=(n, 2)), rng.normal(size=(n, 8))])
beta = np.zeros(X.shape[1]); beta[:5] = 1.0; beta[5] = 2.5
y = X @ beta + rng.normal(0, 1.0, size=n)
names = [f"группа {i+1}" for i in range(5)] + ["сильный", "лишний"] + \
        [f"шум {i+1}" for i in range(8)]

sel_l, sel_e = np.zeros(X.shape[1]), np.zeros(X.shape[1])
for _ in range(R):
    idx = rng.integers(0, n, n)
    Xb = StandardScaler().fit_transform(X[idx]); yb = y[idx]
    sel_l += np.abs(Lasso(alpha=alpha, max_iter=5000).fit(Xb, yb).coef_) > 1e-8
    # у Elastic Net alpha подобран так, чтобы L1-часть штрафа была той же силы
    sel_e += np.abs(ElasticNet(alpha=alpha / l1_ratio, l1_ratio=l1_ratio,
                               max_iter=5000).fit(Xb, yb).coef_) > 1e-8
sel_l, sel_e = sel_l / R, sel_e / R

print(f"средняя корреляция внутри группы: "
      f"{np.corrcoef(group.T)[np.triu_indices(5, 1)].mean():.3f}\n")
print(f"{'признак':<14}{'Lasso':>8}{'Elastic Net':>14}")
for nm, a, b in zip(names[:7], sel_l[:7], sel_e[:7]):
    print(f"{nm:<14}{a:>8.2f}{b:>14.2f}")
print(f"{'макс. по шуму':<14}{sel_l[7:].max():>8.2f}{sel_e[7:].max():>14.2f}")
print(f"\nвнутри группы Lasso: {sel_l[:5].min():.2f}…{sel_l[:5].max():.2f}, "
      f"Elastic Net: {sel_e[:5].min():.2f}…{sel_e[:5].max():.2f}")

средняя корреляция внутри группы: 0.990

признак          Lasso   Elastic Net
группа 1          0.91          1.00
группа 2          0.42          1.00
группа 3          0.81          1.00
группа 4          0.59          1.00
группа 5          0.06          1.00
сильный           1.00          1.00
лишний            0.00          0.00
макс. по шуму     0.00          0.00

внутри группы Lasso: 0.06…0.91, Elastic Net: 1.00…1.00


Один и тот же по смыслу признак попадает в модель то почти всегда, то почти никогда —
в зависимости от того, какая выборка досталась. Отсюда правило stability selection:
доверять не одному прогону, а частоте отбора по подвыборкам.

## 6. Что осталось в руках

- Отбор — часть обучения. Внутри `Pipeline` оценка честная, снаружи — завышенная, и на
  данных без сигнала разрыв достигает четырёх десятых accuracy.
- Корреляция видит только линейную связь; взаимная информация видит любую, но её оценка
  тем шумнее, чем меньше выборка: на малых n чистый шум получает ненулевые оценки.
- Взаимодействие ломает не только одномерный фильтр: жадная обёртка добавляет по одному
  признаку за шаг, и на XOR первый шаг не даёт прироста — она уходит за приманкой и шумом.
- MDI смещена в пользу признаков с большим числом порогов; permutation importance надо
  считать на отложенных данных, иначе она повторит переобучение модели.
- «Признак не выбран» — утверждение о выборке, а не о признаке: внутри группы близнецов
  выбор представителя случаен.